# 🚀 Atlas Architecture: Sovereign AI Deployment
### [Zero-Budget Hardened Edition]

هذا الدفتر هو واجهة التشغيل الرسمية لمشروع **Atlas Capture** على سحابة جوجل. تم تصميمه ليعمل كـ "سيرفر إنتاج" (Production Server) يقوم بمعالجة المهام بذكاء واستقرار عالٍ.

## Step 1: Environment Setup
Mount Google Drive, unpack the lightweight Colab package if it exists, seed `.git` metadata when needed, and install runtime dependencies.


In [ ]:
from google.colab import drive, userdata
import os
import shutil
import subprocess
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ZIP = "/content/drive/MyDrive/Atlas_Automation/Atlas_Project_Colab.zip"
REPO_URL = "https://github.com/aymank2020/OCR_Annotation_Atlas.git"
RUNTIME_DIR = "/content/atlas_runtime"
SEED_DIR = "/content/atlas_repo_seed"

runtime_path = Path(RUNTIME_DIR)
runtime_path.mkdir(parents=True, exist_ok=True)

if Path(PROJECT_ZIP).exists():
    subprocess.run(["unzip", "-o", PROJECT_ZIP, "-d", RUNTIME_DIR], check=True)
    print("Using Drive package as bootstrap artifact.")
else:
    print("Drive package not found; continuing with Git bootstrap only.")

if not (runtime_path / ".git").exists():
    if Path(SEED_DIR).exists():
        shutil.rmtree(SEED_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, SEED_DIR], check=True)
    for item in Path(SEED_DIR).iterdir():
        target = runtime_path / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)
    print("Git metadata seeded into runtime directory.")
else:
    print("Existing Git metadata detected in runtime directory.")

%cd {RUNTIME_DIR}
!pip install -q -r requirements.txt
!playwright install chromium
!playwright install-deps chromium

print("Environment ready.")


## Step 2: Production Update (Git Sync)
Fetch the latest repository state from GitHub and fast-sync the Colab runtime to `origin/main`.


In [ ]:
%cd {RUNTIME_DIR}
!git fetch --all
!git reset --hard origin/main
!pip install -q -r requirements.txt
print("Project synced with latest GitHub main.")


## Step 3: Run Sovereign Autonomous Pipeline
Load secrets from Colab userdata and run the production Colab config.


In [ ]:
import os
from google.colab import userdata

secret_map = {
    "GEMINI_API_KEY": "GEMINI_API_KEY",
    "GEMINI_API_KEY_FALLBACK": "GEMINI_API_KEY_SECONDARY",
    "ATLAS_LOGIN_EMAIL": "ATLAS_LOGIN_EMAIL",
    "ATLAS_EMAIL": "ATLAS_LOGIN_EMAIL",
    "GMAIL_USER": "GMAIL_USER",
    "GMAIL_EMAIL": "GMAIL_USER",
    "GMAIL_APP_PASSWORD": "GMAIL_APP_PASSWORD",
}

for env_key, secret_key in secret_map.items():
    value = userdata.get(secret_key)
    if value:
        os.environ[env_key] = value

!python atlas_web_auto_solver.py --execute --config configs/production_colab.yaml
